In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_data_path = project_root / "data" / "processed"
database_path = processed_data_path / "ravenstack.db"

engine = create_engine(f"sqlite:///{database_path}")

print("Database path:", database_path)
print("Database exists:", database_path.exists())

Database path: c:\Users\tOBESky\Documents\ML_BigDATA\saas-customer-churn-prediction\data\processed\ravenstack.db
Database exists: True


In [3]:
customer_features = pd.read_sql_query(
    """
    SELECT
        account_id,
        industry,
        country,
        signup_date,
        referral_source,
        plan_tier AS initial_plan_tier,
        seats AS initial_seats,
        churn_flag
    FROM accounts;
    """,
    engine
)

customer_features.head()

,account_id,industry,country,signup_date,referral_source,initial_plan_tier,initial_seats,churn_flag
0,A-2e4581,EdTech,US,2024-10-16 00:00:00.000000,partner,Basic,9,0
1,A-43a9e3,FinTech,IN,2023-08-17 00:00:00.000000,other,Basic,18,1
2,A-0a282f,DevTools,US,2024-08-27 00:00:00.000000,organic,Basic,1,0
3,A-1f0ac7,HealthTech,UK,2023-08-27 00:00:00.000000,other,Basic,24,0
4,A-ce550d,HealthTech,US,2024-10-27 00:00:00.000000,event,Enterprise,35,1


In [4]:
print("Rows:", len(customer_features))
print("Unique accounts:", customer_features["account_id"].nunique())
print("Columns:", customer_features.shape[1])

Rows: 500
Unique accounts: 500
Columns: 8


In [5]:
subscription_features = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    subscription_history AS (
        SELECT
            account_id,
            COUNT(*) AS subscription_count,
            MAX(upgrade_flag) AS ever_upgraded,
            MAX(downgrade_flag) AS ever_downgraded
        FROM subscriptions
        GROUP BY account_id
    ),

    latest_subscription AS (
        SELECT
            account_id,
            plan_tier AS latest_plan_tier,
            seats AS latest_seats,
            mrr_amount AS latest_mrr,
            is_trial AS latest_is_trial,
            billing_frequency AS latest_billing_frequency,
            auto_renew_flag AS latest_auto_renew
        FROM ranked_subscriptions
        WHERE row_number = 1
    )

    SELECT
        h.account_id,
        h.subscription_count,
        h.ever_upgraded,
        h.ever_downgraded,
        l.latest_plan_tier,
        l.latest_seats,
        l.latest_mrr,
        l.latest_is_trial,
        l.latest_billing_frequency,
        l.latest_auto_renew

    FROM subscription_history AS h

    JOIN latest_subscription AS l
        ON h.account_id = l.account_id;
    """,
    engine
)

subscription_features.head()

,account_id,subscription_count,ever_upgraded,ever_downgraded,latest_plan_tier,latest_seats,latest_mrr,latest_is_trial,latest_billing_frequency,latest_auto_renew
0,A-00bed1,10,1,0,Enterprise,28,5572,0,annual,0
1,A-00cac8,9,0,1,Pro,19,931,0,annual,1
2,A-0158bb,6,1,0,Pro,45,2205,0,annual,1
3,A-016043,11,1,0,Enterprise,13,2587,0,annual,1
4,A-019782,9,1,0,Enterprise,15,0,1,annual,0


In [6]:
customer_features = customer_features.merge(
    subscription_features,
    on="account_id",
    how="left"
)

In [7]:
print("Shape after subscription merge:", customer_features.shape)
print("Unique accounts:", customer_features["account_id"].nunique())

customer_features.head()

Shape after subscription merge: (500, 17)
Unique accounts: 500


,account_id,industry,country,signup_date,referral_source,initial_plan_tier,initial_seats,churn_flag,subscription_count,ever_upgraded,ever_downgraded,latest_plan_tier,latest_seats,latest_mrr,latest_is_trial,latest_billing_frequency,latest_auto_renew
0,A-2e4581,EdTech,US,2024-10-16 00:00:00.000000,partner,Basic,9,0,10,1,0,Basic,44,836,0,monthly,1
1,A-43a9e3,FinTech,IN,2023-08-17 00:00:00.000000,other,Basic,18,1,8,1,0,Pro,18,882,0,monthly,1
2,A-0a282f,DevTools,US,2024-08-27 00:00:00.000000,organic,Basic,1,0,15,1,1,Pro,2,98,0,annual,1
3,A-1f0ac7,HealthTech,UK,2023-08-27 00:00:00.000000,other,Basic,24,0,7,1,0,Pro,24,1176,0,monthly,1
4,A-ce550d,HealthTech,US,2024-10-27 00:00:00.000000,event,Enterprise,35,1,9,1,0,Enterprise,109,21691,0,monthly,1


In [8]:
print("Shape:", customer_features.shape)
print("Rows:", len(customer_features))
print("Unique accounts:", customer_features["account_id"].nunique())

Shape: (500, 17)
Rows: 500
Unique accounts: 500


In [9]:
usage_features = pd.read_sql_query(
    """
    SELECT
        s.account_id,

        COUNT(*) AS usage_event_count,

        SUM(f.usage_count) AS total_usage_count,

        SUM(f.usage_duration_secs) AS total_usage_duration_secs,

        SUM(f.error_count) AS total_errors,

        COUNT(DISTINCT f.feature_name) AS unique_features_used,

        COUNT(DISTINCT DATE(f.usage_date)) AS active_usage_days,

        MAX(f.usage_date) AS last_usage_date,

        SUM(
            CASE
                WHEN f.is_beta_feature = 1 THEN 1
                ELSE 0
            END
        ) AS beta_usage_events

    FROM feature_usage AS f

    JOIN subscriptions AS s
        ON f.subscription_id = s.subscription_id

    GROUP BY s.account_id;
    """,
    engine
)

usage_features.head()

,account_id,usage_event_count,total_usage_count,total_usage_duration_secs,total_errors,unique_features_used,active_usage_days,last_usage_date,beta_usage_events
0,A-00bed1,51,514,143734,27,32,49,2024-12-14 00:00:00.000000,2
1,A-00cac8,58,602,171366,31,30,57,2024-12-24 00:00:00.000000,7
2,A-0158bb,36,364,122051,22,19,35,2024-12-28 00:00:00.000000,3
3,A-016043,47,490,132075,21,26,46,2024-12-31 00:00:00.000000,3
4,A-019782,55,562,160848,30,28,55,2024-12-20 00:00:00.000000,3


In [10]:
print("Rows:", len(usage_features))
print(
    "Unique accounts:",
    usage_features["account_id"].nunique()
)

Rows: 500
Unique accounts: 500


In [11]:
customer_features = customer_features.merge(
    usage_features,
    on="account_id",
    how="left"
)

In [12]:
print("Shape:", customer_features.shape)
print("Rows:", len(customer_features))
print(
    "Unique accounts:",
    customer_features["account_id"].nunique()
)

Shape: (500, 25)
Rows: 500
Unique accounts: 500


In [13]:
customer_features["signup_date"] = pd.to_datetime(
    customer_features["signup_date"]
)

customer_features["last_usage_date"] = pd.to_datetime(
    customer_features["last_usage_date"]
)

In [14]:
snapshot_date = customer_features["last_usage_date"].max()

print("Snapshot date:", snapshot_date)

Snapshot date: 2024-12-31 00:00:00


In [15]:
customer_features["errors_per_100_usage"] = (
    customer_features["total_errors"]
    / customer_features["total_usage_count"]
    * 100
)

In [17]:
customer_features["average_duration_per_event"] = (
    customer_features["total_usage_duration_secs"]
    / customer_features["usage_event_count"]
)

In [18]:
customer_features["average_usage_per_event"] = (
    customer_features["total_usage_count"]
    / customer_features["usage_event_count"]
)

In [24]:
customer_features[
    [
        "account_id",
        "usage_event_count",
        "total_usage_count",
        "total_errors",
        "errors_per_100_usage",
        "average_usage_per_event",
        "average_duration_per_event",
        "days_since_last_usage"
    ]
].head()

,account_id,usage_event_count,total_usage_count,total_errors,errors_per_100_usage,average_usage_per_event,average_duration_per_event,days_since_last_usage
0,A-2e4581,55,535,38,7.102804,9.727273,2769.800000,21
1,A-43a9e3,35,355,14,3.943662,10.142857,2889.600000,2
2,A-0a282f,83,821,48,5.846529,9.891566,3026.626506,9
3,A-1f0ac7,41,382,21,5.497382,9.317073,2500.682927,20
4,A-ce550d,58,579,31,5.354059,9.982759,3720.327586,8


In [21]:
customer_features.columns.tolist()

['account_id',
 'industry',
 'country',
 'signup_date',
 'referral_source',
 'initial_plan_tier',
 'initial_seats',
 'churn_flag',
 'subscription_count',
 'ever_upgraded',
 'ever_downgraded',
 'latest_plan_tier',
 'latest_seats',
 'latest_mrr',
 'latest_is_trial',
 'latest_billing_frequency',
 'latest_auto_renew',
 'usage_event_count',
 'total_usage_count',
 'total_usage_duration_secs',
 'total_errors',
 'unique_features_used',
 'active_usage_days',
 'last_usage_date',
 'beta_usage_events',
 'errors_per_100_usage',
 'average_duration_per_event',
 'average_usage_per_event']

In [22]:
customer_features["last_usage_date"] = pd.to_datetime(
    customer_features["last_usage_date"]
)

snapshot_date = customer_features["last_usage_date"].max()

customer_features["days_since_last_usage"] = (
    snapshot_date - customer_features["last_usage_date"]
).dt.days

In [23]:
print("Snapshot date:", snapshot_date)

customer_features[
    ["account_id", "last_usage_date", "days_since_last_usage"]
].head()

Snapshot date: 2024-12-31 00:00:00


,account_id,last_usage_date,days_since_last_usage
0,A-2e4581,2024-12-10,21
1,A-43a9e3,2024-12-29,2
2,A-0a282f,2024-12-22,9
3,A-1f0ac7,2024-12-11,20
4,A-ce550d,2024-12-23,8


In [25]:
support_features = pd.read_sql_query(
    """
    SELECT
        account_id,

        COUNT(*) AS ticket_count,

        AVG(first_response_time_minutes)
            AS average_first_response_minutes,

        AVG(resolution_time_hours)
            AS average_resolution_hours,

        AVG(satisfaction_score)
            AS average_satisfaction_score,

        SUM(escalation_flag)
            AS escalation_count,

        MAX(submitted_at)
            AS last_support_ticket_date

    FROM support_tickets

    GROUP BY account_id;
    """,
    engine
)

support_features.head()

,account_id,ticket_count,average_first_response_minutes,average_resolution_hours,average_satisfaction_score,escalation_count,last_support_ticket_date
0,A-00bed1,4,106.25,31.750000,4.0,0,2024-05-16 00:00:00.000000
1,A-00cac8,2,120.50,33.000000,NaN,0,2023-09-15 00:00:00.000000
2,A-0158bb,1,50.00,32.000000,3.0,0,2024-01-11 00:00:00.000000
3,A-016043,3,78.00,30.333333,4.0,0,2024-12-11 00:00:00.000000
4,A-019782,2,107.00,10.000000,3.0,0,2024-12-01 00:00:00.000000


In [26]:
print("Rows:", len(support_features))
print(
    "Unique accounts:",
    support_features["account_id"].nunique()
)

Rows: 492
Unique accounts: 492


In [27]:
customer_features = customer_features.merge(
    support_features,
    on="account_id",
    how="left"
)

In [28]:
print("Shape after support merge:", customer_features.shape)
print("Rows:", len(customer_features))
print(
    "Unique accounts:",
    customer_features["account_id"].nunique()
)

Shape after support merge: (500, 35)
Rows: 500
Unique accounts: 500


In [29]:
customer_features["ticket_count"] = (
    customer_features["ticket_count"].fillna(0)
)

customer_features["escalation_count"] = (
    customer_features["escalation_count"].fillna(0)
)

In [30]:
customer_features["has_support_tickets"] = (
    customer_features["ticket_count"] > 0
).astype(int)

In [31]:
customer_features[
    [
        "account_id",
        "ticket_count",
        "has_support_tickets",
        "average_first_response_minutes",
        "average_resolution_hours",
        "average_satisfaction_score",
        "escalation_count"
    ]
].head(10)

,account_id,ticket_count,has_support_tickets,average_first_response_minutes,average_resolution_hours,average_satisfaction_score,escalation_count
0,A-2e4581,2.0,1,91.000000,23.000000,3.000000,0.0
1,A-43a9e3,3.0,1,73.333333,38.000000,4.000000,0.0
2,A-0a282f,3.0,1,63.666667,43.666667,4.666667,0.0
3,A-1f0ac7,2.0,1,174.000000,29.000000,NaN,0.0
4,A-ce550d,7.0,1,107.857143,42.285714,3.800000,1.0
5,A-1b9609,4.0,1,74.750000,43.000000,3.000000,0.0
6,A-a0ca4e,6.0,1,89.833333,39.166667,3.800000,3.0
7,A-e5d6ab,3.0,1,57.000000,42.666667,NaN,0.0
8,A-7dacce,4.0,1,75.250000,32.500000,3.666667,0.0
9,A-10b8da,3.0,1,71.333333,31.333333,3.500000,0.0


In [32]:
customer_features["signup_date"] = pd.to_datetime(
    customer_features["signup_date"]
)

customer_features["customer_tenure_days"] = (
    snapshot_date - customer_features["signup_date"]
).dt.days

In [33]:
customer_features["seat_change"] = (
    customer_features["latest_seats"]
    - customer_features["initial_seats"]
)

In [34]:
customer_features["plan_changed"] = (
    customer_features["initial_plan_tier"]
    != customer_features["latest_plan_tier"]
).astype(int)

In [35]:
customer_features["last_support_ticket_date"] = pd.to_datetime(
    customer_features["last_support_ticket_date"]
)

In [36]:
customer_features["days_since_last_support_ticket"] = (
    snapshot_date - customer_features["last_support_ticket_date"]
).dt.days

In [37]:
customer_features[
    [
        "account_id",
        "customer_tenure_days",
        "initial_seats",
        "latest_seats",
        "seat_change",
        "initial_plan_tier",
        "latest_plan_tier",
        "plan_changed",
        "days_since_last_usage",
        "days_since_last_support_ticket",
        "churn_flag"
    ]
].head()

,account_id,customer_tenure_days,initial_seats,latest_seats,seat_change,initial_plan_tier,latest_plan_tier,plan_changed,days_since_last_usage,days_since_last_support_ticket,churn_flag
0,A-2e4581,76,9,44,35,Basic,Basic,0,21,21.0,0
1,A-43a9e3,502,18,18,0,Basic,Pro,1,2,188.0,1
2,A-0a282f,126,1,2,1,Basic,Pro,1,9,66.0,0
3,A-1f0ac7,492,24,24,0,Basic,Pro,1,20,263.0,0
4,A-ce550d,65,35,109,74,Enterprise,Enterprise,0,8,67.0,1


In [38]:
missing_values = (
    customer_features
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_values[missing_values > 0]

average_satisfaction_score        34
last_support_ticket_date           8
average_resolution_hours           8
average_first_response_minutes     8
days_since_last_support_ticket     8
dtype: int64

In [39]:
# 1. Shape / grain
print("Final shape:", customer_features.shape)
print("Total rows:", len(customer_features))
print("Unique accounts:", customer_features["account_id"].nunique())

# 2. Columns
print(customer_features.columns.tolist())

# 3. Missing values
missing_values = (
    customer_features
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

print(missing_values[missing_values > 0])

# 4. Target distribution
print(customer_features["churn_flag"].value_counts())
print(
    customer_features["churn_flag"]
    .value_counts(normalize=True)
    .round(3)
)

Final shape: (500, 40)
Total rows: 500
Unique accounts: 500
['account_id', 'industry', 'country', 'signup_date', 'referral_source', 'initial_plan_tier', 'initial_seats', 'churn_flag', 'subscription_count', 'ever_upgraded', 'ever_downgraded', 'latest_plan_tier', 'latest_seats', 'latest_mrr', 'latest_is_trial', 'latest_billing_frequency', 'latest_auto_renew', 'usage_event_count', 'total_usage_count', 'total_usage_duration_secs', 'total_errors', 'unique_features_used', 'active_usage_days', 'last_usage_date', 'beta_usage_events', 'errors_per_100_usage', 'average_duration_per_event', 'average_usage_per_event', 'days_since_last_usage', 'ticket_count', 'average_first_response_minutes', 'average_resolution_hours', 'average_satisfaction_score', 'escalation_count', 'last_support_ticket_date', 'has_support_tickets', 'customer_tenure_days', 'seat_change', 'plan_changed', 'days_since_last_support_ticket']
average_satisfaction_score        34
last_support_ticket_date           8
average_resolution_h

In [40]:
customer_features["has_satisfaction_score"] = (
    customer_features["average_satisfaction_score"]
    .notna()
    .astype(int)
)

In [41]:
feature_columns = [
    # Account characteristics
    "industry",
    "country",
    "referral_source",
    "initial_plan_tier",
    "customer_tenure_days",

    # Subscription behaviour
    "subscription_count",
    "ever_upgraded",
    "ever_downgraded",
    "latest_plan_tier",
    "latest_seats",
    "seat_change",
    "latest_mrr",
    "latest_is_trial",
    "latest_billing_frequency",
    "latest_auto_renew",
    "plan_changed",

    # Product usage
    "usage_event_count",
    "total_usage_count",
    "unique_features_used",
    "active_usage_days",
    "beta_usage_events",
    "errors_per_100_usage",
    "average_duration_per_event",
    "average_usage_per_event",
    "days_since_last_usage",

    # Support behaviour
    "ticket_count",
    "has_support_tickets",
    "average_first_response_minutes",
    "average_resolution_hours",
    "average_satisfaction_score",
    "has_satisfaction_score",
    "escalation_count",
    "days_since_last_support_ticket"
]

In [42]:
X = customer_features[feature_columns].copy()

y = customer_features["churn_flag"].copy()

In [43]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(
    y.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

X shape: (500, 33)
y shape: (500,)

Target distribution:
churn_flag
0    390
1    110
Name: count, dtype: int64

Target percentage:
churn_flag
0    78.0
1    22.0
Name: proportion, dtype: float64
